# 🤖 06: Agentic Search & Tool Use

---

| 항목 | 내용 |
|------|------|
| **목표** | 단발성 검색이 아니라, 모델이 도구를 반복 사용하며 복잡한 질문을 해결하는 흐름을 이해한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 7분 / 전체 18분 |
| **API 키** | ✅ 권장 (없으면 rule-based 플래너 + mock 응답으로 흐름 시연) |
| **이전 노트북과의 연결** | 검색 품질을 높였다. 하지만 한 번 검색으로 끝내지 않고, 모델이 반복적으로 도구를 쓰는 방식으로 확장한다. |

---

## 🎯 핵심 메시지

> **실전 AI 시스템은 단발성 RAG보다 Agentic Retrieval 쪽으로 진화하고 있습니다.**  
> **LLM이 도구를 직접 선택하고, 결과에 따라 다음 도구를 결정합니다.**

```
단순 RAG:
  질문 → 검색 1회 → LLM → 답변

Agentic Search:
  질문 → LLM이 도구 선택 → 도구 실행 → 결과 검토
         → 추가 도구 필요? → 도구 실행 → ...
         → 충분히 수집됨 → 최종 답변
```

In [ ]:
!pip install -q openai sentence-transformers rank-bm25
print("✅ 완료")

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from typing import List, Dict, Optional

# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(key=""):
    k = key or os.environ.get("OPENAI_API_KEY", "")
    if k and k not in ("", "sk-..."):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=k)
            print("✅ API 모드 (Tool Use 포함)")
            return "api", c
        except: pass
    print("💡 로컬 모드 - rule-based 플래너 + mock 응답")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

_st = None
def embed(texts):
    global _st
    if MODE == "api" and client:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                r = client.embeddings.create(input=texts[i:i+50], model="text-embedding-3-small")
                vecs.extend([x.embedding for x in r.data])
            return np.array(vecs, dtype=np.float32)
        except: pass
    if _st is None:
        print("📥 임베딩 모델 로딩...")
        from sentence_transformers import SentenceTransformer
        _st = SentenceTransformer("all-MiniLM-L6-v2")
        print("✅")
    return _st.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

def cos_sim(q, D):
    q = np.array(q, dtype=np.float32).flatten()
    D = np.array(D, dtype=np.float32)
    qn = np.linalg.norm(q)
    if qn < 1e-9: return np.zeros(len(D))
    dn = np.linalg.norm(D, axis=1)
    dn = np.where(dn < 1e-9, 1e-9, dn)
    return (D @ q) / (dn * qn)

def call_llm(prompt, system="당신은 테크코어 내부 AI 어시스턴트입니다.",
             mock=None, temperature=0.3):
    if MODE == "api" and client:
        try:
            r = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"system","content":system},{"role":"user","content":prompt}],
                temperature=temperature, max_tokens=800
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            print(f"⚠️ {e}")
    return f"[Mock]\n{mock}" if mock else "[로컬 모드]"

print(f"모드: {MODE}")

## 1️⃣ 도구(Tool) 정의

에이전트가 사용할 수 있는 도구들을 정의합니다.  
실제 서비스에서는 이 도구들이 DB 쿼리, 외부 API 호출 등과 연결됩니다.

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# ──────────────────────────────────────────────────────
# 문서 데이터베이스 (도구들의 데이터 소스)
# ──────────────────────────────────────────────────────
from helpers.sample_data import RELEASE_NOTES, POLICIES, SAMPLE_DOCS_06 as DOCS
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print("📊 문서 벡터 인덱스 구축 중...")
DOC_VECS = embed([d["content"] for d in DOCS])
print(f"✅ 완료 ({len(DOCS)}개 문서)")

In [ ]:
# ──────────────────────────────────────────────────────
# 도구 함수 정의
# ──────────────────────────────────────────────────────

tool_call_log = []  # 호출 기록 (교육용 시각화)

def search_docs(query: str, category: str = None, top_k: int = 3) -> List[Dict]:
    """
    [도구 1] 내부 문서를 벡터 검색합니다.
    
    Args:
        query: 검색 쿼리
        category: 문서 카테고리 필터 (선택)
        top_k: 반환할 문서 수
    """
    tool_call_log.append({"tool": "search_docs", "args": {"query": query, "category": category}})
    print(f"    🔧 [search_docs] query='{query}' category={category}")

    q_vec = embed([query])[0]
    scores = cos_sim(q_vec, DOC_VECS)

    results = []
    for i, score in enumerate(np.argsort(scores)[::-1]):
        doc = DOCS[score]
        if category and doc["category"] != category:
            continue
        results.append({
            "doc_id": doc["doc_id"],
            "content": doc["content"],
            "similarity": float(scores[score])
        })
        if len(results) >= top_k:
            break

    print(f"    → 검색 결과: {[r['doc_id'] for r in results]}")
    return results


def lookup_release_note(product: str, version: str = "latest") -> str:
    """
    [도구 2] 제품 릴리즈 노트를 조회합니다.
    
    Args:
        product: 제품명 (CloudSync, DataPulse)
        version: 버전 (latest, v2.3, v1.1 등)
    """
    tool_call_log.append({"tool": "lookup_release_note", "args": {"product": product, "version": version}})
    print(f"    🔧 [lookup_release_note] product='{product}' version='{version}'")

    product_data = RELEASE_NOTES.get(product, {})
    if version == "latest":
        version = product_data.get("latest", "")

    result = product_data.get(version, f"{product} {version} 정보를 찾을 수 없습니다.")
    print(f"    → 결과: {result[:60]}...")
    return result


def lookup_policy(topic: str) -> str:
    """
    [도구 3] 내부 정책을 주제로 조회합니다.
    
    Args:
        topic: 정책 주제 (보안, 원격근무, 배포, 인시던트 등)
    """
    tool_call_log.append({"tool": "lookup_policy", "args": {"topic": topic}})
    print(f"    🔧 [lookup_policy] topic='{topic}'")

    # 가장 유사한 정책 키 찾기
    for key in POLICIES:
        if topic.lower() in key.lower() or key.lower() in topic.lower():
            result = POLICIES[key]
            print(f"    → 결과: {result[:60]}...")
            return result

    # 벡터 유사도로 fallback
    policy_keys = list(POLICIES.keys())
    policy_vecs = embed(policy_keys)
    query_vec = embed([topic])[0]
    best_idx = int(np.argmax(cos_sim(query_vec, policy_vecs)))
    result = POLICIES[policy_keys[best_idx]]
    print(f"    → 결과 (유사도 기반): {result[:60]}...")
    return result


# 도구 레지스트리
TOOLS = {
    "search_docs": search_docs,
    "lookup_release_note": lookup_release_note,
    "lookup_policy": lookup_policy,
}

print("✅ 도구 정의 완료")
print("  사용 가능한 도구:")
for name, fn in TOOLS.items():
    print(f"  • {name}: {fn.__doc__.strip().split(chr(10))[0]}")

## 2️⃣ Agentic 플래너 구현

두 가지 방식을 구현합니다:
- **Rule-based Planner**: 쿼리 분석 → 도구 호출 계획 (API 없이 작동)
- **LLM Planner**: LLM이 직접 도구를 선택 (API 있을 때)

In [ ]:
def rule_based_plan(query: str) -> List[Dict]:
    """
    Rule-based 플래너: 쿼리 분석 → 도구 호출 계획.
    LLM 없이도 동작하는 fallback 방식.
    """
    q_lower = query.lower()
    plan = []

    # 릴리즈 노트 관련
    products_mentioned = []
    if "cloudsync" in q_lower:
        products_mentioned.append("CloudSync")
    if "datapulse" in q_lower:
        products_mentioned.append("DataPulse")

    for product in products_mentioned:
        version = "latest"
        for v in ["v2.3", "v2.2", "v1.1", "v1.0"]:
            if v in query:
                version = v
                break
        plan.append({
            "tool": "lookup_release_note",
            "args": {"product": product, "version": version},
            "reason": f"{product} 릴리즈 노트 조회 (버전: {version})"
        })

    # 정책 관련
    policy_keywords = {"보안": "보안", "api 키": "API 키 관리", "인증": "보안",
                       "재택": "원격근무", "원격": "원격근무",
                       "배포": "배포", "장애": "인시던트", "인시던트": "인시던트"}
    for kw, topic in policy_keywords.items():
        if kw in q_lower:
            if not any(p["tool"] == "lookup_policy" and p["args"]["topic"] == topic
                       for p in plan):
                plan.append({
                    "tool": "lookup_policy",
                    "args": {"topic": topic},
                    "reason": f"'{topic}' 정책 조회 (키워드: {kw})"
                })

    # 일반 문서 검색 (항상 포함)
    plan.append({
        "tool": "search_docs",
        "args": {"query": query, "top_k": 3},
        "reason": "관련 내부 문서 벡터 검색"
    })

    return plan


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "내부 문서를 의미 기반으로 검색합니다. 정책, FAQ, 회의록 등 일반 문서 검색에 사용.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색 쿼리"},
                    "category": {"type": "string", "enum": ["보안정책","릴리즈노트","HR정책","FAQ","프로세스","회의록"], "description": "문서 카테고리 (선택)"},
                    "top_k": {"type": "integer", "description": "반환할 문서 수 (기본 3)"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_release_note",
            "description": "제품 릴리즈 노트를 버전별로 조회합니다. 특정 버전의 변경사항, Breaking Changes 확인에 사용.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {"type": "string", "enum": ["CloudSync", "DataPulse"], "description": "제품명"},
                    "version": {"type": "string", "description": "버전 (latest, v2.3, v1.1 등)"}
                },
                "required": ["product"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_policy",
            "description": "내부 정책을 주제로 조회합니다. 보안, 원격근무, 배포, 인시던트 등의 정책 확인에 사용.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "정책 주제"}
                },
                "required": ["topic"]
            }
        }
    }
]

print("✅ 플래너 준비 완료")

## 3️⃣ Agentic 파이프라인 실행

In [ ]:
def run_agentic_pipeline(query: str, max_steps: int = 5):
    """
    Agentic Search 파이프라인:
    1. LLM이 도구 호출 계획 결정 (API 모드) 또는 rule-based 플래너 (로컬 모드)
    2. 도구 실행
    3. 수집된 정보로 최종 답변 생성
    """
    global tool_call_log
    tool_call_log = []

    print(f"\n{'='*60}")
    print(f"  🤖 Agentic Search 시작")
    print(f"  질문: '{query}'")
    print(f"{'='*60}")

    collected_info = []

    if MODE == "api" and client:
        # ─────────────────────────────────────────
        # LLM이 직접 도구를 선택하는 방식
        # ─────────────────────────────────────────
        messages = [
            {"role": "system", "content": "당신은 테크코어 내부 지식 어시스턴트입니다. 사용 가능한 도구를 활용하여 정확한 정보를 수집하고 답변하세요."},
            {"role": "user", "content": query}
        ]

        for step in range(max_steps):
            print(f"\n[Step {step+1}] LLM 도구 선택 중...")
            try:
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=messages,
                    tools=TOOL_DEFINITIONS,
                    tool_choice="auto",
                    max_tokens=500
                )
                msg = response.choices[0].message

                if not msg.tool_calls:  # 도구 호출 없음 = 최종 답변
                    print(f"\n[Step {step+1}] LLM이 충분히 정보를 수집했다고 판단. 최종 답변 생성.")
                    final_answer = msg.content
                    break

                messages.append(msg)

                for tc in msg.tool_calls:
                    tool_name = tc.function.name
                    args = json.loads(tc.function.arguments)
                    print(f"  🔧 [{tool_name}] args={args}")

                    if tool_name in TOOLS:
                        result = TOOLS[tool_name](**args)
                        result_str = json.dumps(result, ensure_ascii=False) if isinstance(result, list) else str(result)
                        collected_info.append({"tool": tool_name, "args": args, "result": result_str[:200]})

                        messages.append({
                            "role": "tool",
                            "tool_call_id": tc.id,
                            "content": result_str
                        })
            except Exception as e:
                print(f"  ⚠️ Step {step+1} 오류: {e}")
                break
        else:
            final_answer = "최대 단계 초과 - 수집된 정보로 답변합니다."

    else:
        # ─────────────────────────────────────────
        # Rule-based 플래너 (로컬 모드)
        # ─────────────────────────────────────────
        plan = rule_based_plan(query)
        print(f"\n[Rule-based 플래너] 실행 계획 ({len(plan)}개 도구):")
        for i, step in enumerate(plan, 1):
            print(f"  {i}. {step['tool']}({step['args']}) - {step['reason']}")

        print("\n도구 실행:")
        for step in plan:
            tool_fn = TOOLS.get(step["tool"])
            if tool_fn:
                result = tool_fn(**step["args"])
                result_str = json.dumps(result, ensure_ascii=False) if isinstance(result, list) else str(result)
                collected_info.append({
                    "tool": step["tool"],
                    "args": step["args"],
                    "result": result_str[:200]
                })

        # 수집된 정보로 최종 답변 생성
        context = "\n\n".join(
            f"[{info['tool']} 결과]\n{info['result']}" for info in collected_info
        )
        answer_prompt = f"""아래 도구 실행 결과를 종합하여 질문에 답변하세요.

질문: {query}

수집된 정보:
{context}

종합 답변:"""

        final_answer = call_llm(answer_prompt)

    print(f"\n{'─'*60}")
    print(f"📋 최종 답변:")
    print(final_answer)
    print(f"\n📊 총 도구 호출 수: {len(tool_call_log)}회")

    return {
        "query": query,
        "tool_calls": tool_call_log,
        "collected_info": collected_info,
        "answer": final_answer
    }

print("✅ Agentic 파이프라인 준비 완료")

## 4️⃣ 데모: 단발성 RAG vs Agentic Search 비교

In [ ]:
# ─────────────────────────────────────────────────────────
# 복잡한 질문 1: 보안 정책 변경 + CloudSync 배포 일정 통합
# 단발성 검색 1회로는 두 가지를 함께 잡기 어렵습니다.
# ─────────────────────────────────────────────────────────

COMPLEX_QUERY_1 = "보안 정책 v3.1 변경 사항과 CloudSync v2.3 배포 일정을 함께 정리해줘"

print("="*60)
print("  📝 단발성 RAG로 시도")
print("="*60)
print(f"  쿼리: '{COMPLEX_QUERY_1}'")
print()

# 단발성 검색 - top 3만 가져오면?
q_vec = embed([COMPLEX_QUERY_1])[0]
scores = cos_sim(q_vec, np.array([embed([d['content']])[0] for d in DOCS]))
top3_idx = np.argsort(scores)[::-1][:3]
simple_results = [DOCS[i]["doc_id"] for i in top3_idx]

print(f"  단발성 검색 결과 (top-3): {simple_results}")

# 이상적인 결과에는 SEC-POL-003 + REL-CLOUDSYNC-023 + MTG-2024Q3-STRATEGY가 있어야 함
ideal = {"SEC-POL-003", "REL-CLOUDSYNC-023", "MTG-2024Q3-STRATEGY"}
found = set(simple_results) & ideal
print(f"  필요한 문서: {ideal}")
print(f"  찾은 문서: {found} {'✅' if len(found) >= 2 else '⚠️ 일부 누락 가능성'}")

In [ ]:
# Agentic Search 실행
result1 = run_agentic_pipeline(COMPLEX_QUERY_1)

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# Mock 답변 (로컬 모드 시연용)
from helpers.sample_data import MOCK_AGENTIC_ANSWER_1
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

if MODE == "local":
    print("\n--- Mock 최종 답변 (API 모드에서는 실제 LLM 응답) ---")
    print(MOCK_AGENTIC_ANSWER_1)

In [ ]:
# 복잡한 질문 2: 릴리즈 노트 + FAQ 통합
COMPLEX_QUERY_2 = "DataPulse v1.1 업그레이드 시 주의사항과 Slack 알림 설정 방법을 같이 정리해줘"
result2 = run_agentic_pipeline(COMPLEX_QUERY_2)

In [ ]:
# 복잡한 질문 3: 여러 주제 통합
COMPLEX_QUERY_3 = "CloudSync v2.3에서 JWT 인증이 제거됐는데, 현행 API 키 보안 정책과 함께 개발팀에 공유할 migration action item을 정리해줘"
result3 = run_agentic_pipeline(COMPLEX_QUERY_3)

## 5️⃣ Agentic vs 단발성 RAG 차이 시각화

In [ ]:
display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:820px;margin:10px auto;">
  <h3 style="color:#2c3e50;">단발성 RAG vs Agentic Search</h3>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <thead>
      <tr style="background:#2c3e50;color:white;">
        <th style="padding:10px;">항목</th>
        <th style="padding:10px;">단발성 RAG</th>
        <th style="padding:10px;">Agentic Search</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">검색 횟수</td>
        <td style="padding:10px;">1회 고정</td>
        <td style="padding:10px;">필요에 따라 N회</td>
      </tr>
      <tr>
        <td style="padding:10px;">도구 종류</td>
        <td style="padding:10px;">벡터 검색 1가지</td>
        <td style="padding:10px;">여러 도구 (검색, 릴리즈노트, 정책 조회 등)</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">적합한 질문</td>
        <td style="padding:10px;">단일 주제, 단순 사실 조회</td>
        <td style="padding:10px;">복수 주제, 비교, 통합 요약</td>
      </tr>
      <tr>
        <td style="padding:10px;">레이턴시</td>
        <td style="padding:10px;">빠름 (1-2초)</td>
        <td style="padding:10px;">느림 (5-30초, 도구 수에 비례)</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">비용</td>
        <td style="padding:10px;">낮음</td>
        <td style="padding:10px;">높음 (LLM 다회 호출)</td>
      </tr>
      <tr>
        <td style="padding:10px;">답변 품질</td>
        <td style="padding:10px;">단순 질문에서 충분</td>
        <td style="padding:10px;">복잡한 질문에서 우수</td>
      </tr>
    </tbody>
  </table>
  <div style="margin-top:12px;padding:12px;background:#f0f4ff;border-left:4px solid #6c5ce7;font-size:13px;">
    🏗️ <strong>실무 선택 가이드</strong><br>
    • 80%의 질문 → 단발성 RAG로 충분<br>
    • 비교/통합/다단계 추론 → Agentic Search<br>
    • 실제 시스템: 라우터로 두 방식을 자동 선택하는 패턴 사용
  </div>
</div>
"""))

---

## 🎤 강의자 멘트 포인트

> **"'보안 정책 변경사항과 CloudSync 배포 일정을 함께 정리해줘'라는 질문을**  
> **단발성 RAG로 해결하려면 벡터 검색 한 번으로 두 주제를 동시에 잡아야 합니다.**  
> **이건 어렵습니다. 두 주제를 동시에 잘 담은 문서가 없으니까요.**  
>
> Agentic Search는 다릅니다.  
> 먼저 보안 정책을 찾고, 그 다음 릴리즈 노트를 찾고,  
> 마지막으로 전략 회의록을 찾아서 세 개를 종합합니다.  
> 이게 요즘 AI 시스템이 진화하는 방향입니다."

## 🙋 청중 질문 유도
> - "Agentic Search의 '최대 반복 횟수'는 왜 필요할까요? 무한 루프가 생길 수 있나요?"
> - "LLM이 도구를 잘못 선택하면 어떻게 될까요? 어떻게 방어할 수 있을까요?"
> - "여러분이 만든다면, 어떤 도구를 추가하고 싶나요?"

## 🏗️ 실무 확장 포인트
- **ReAct 패턴**: Reasoning + Acting 교차 (생각 → 행동 → 관찰 → 다시 생각)
- **Memory**: 이전 대화 컨텍스트 유지 (conversation history)
- **Streaming**: 도구 실행 결과를 실시간으로 사용자에게 표시
- **Safety**: 도구 호출 전 권한 확인, 위험 도구 승인 요청
- **Observability**: LangSmith, Langfuse 등으로 agent 실행 추적

## ➕ 추가 실험
1. 새 도구 `lookup_meeting_notes(date_range)` 추가해보기
2. 도구 정의에서 description을 모호하게 바꿔서 LLM 선택이 어떻게 달라지는지 확인
3. `max_steps=1`로 제한하면 복잡한 질문에서 어떻게 실패하는지 확인

## ➡️ 다음 노트북 (선택)
**07_graphrag_concept_demo.ipynb** - 관계가 중요한 문제에서 구조화된 context의 가치